# SIGEC - Entrenamiento en Kaggle

Entrena neumonia (MobileNetV2) e inasistencia (Random Forest). Los datasets se adjuntan al notebook: **no hace falta kaggle.json**.

**Antes de ejecutar (panel derecho):**
1. *Settings -> Accelerator*: **GPU T4 x2** o P100. *Internet*: **On** (hace falta para pesos ImageNet y `pip`; Kaggle pide verificar el telefono una vez).
2. *Add Input -> Datasets*: busca y agrega
   - `Chest X-Ray Images (Pneumonia)` (paultimothymooney)
   - `Medical Appointment No Shows` (joniarroba)
3. *Add Input -> Upload*: crea un dataset **privado** con `sigec-ml.zip` (scripts de entrenamiento, ~20 KB).

> Proyecto academico. El modelo de neumonia es apoyo a la decision, no un diagnostico.

In [ ]:
import glob, os, shutil
# 0. Que hay adjunto (profundidad 3)
for d, ds, fs in os.walk('/kaggle/input'):
    if d.count('/') - 3 <= 2:
        print(d, '|', ds[:6], '|', [f for f in fs][:4])

# 1. Codigo
train_py = glob.glob('/kaggle/input/**/ml/pneumonia/train.py', recursive=True)
assert train_py, 'No encuentro sigec-ml: agrega el dataset con el zip (Add Input -> Upload)'
SRC = train_py[0].split('/ml/pneumonia/')[0]
if os.path.exists('/kaggle/working/sigec'): shutil.rmtree('/kaggle/working/sigec')
shutil.copytree(SRC, '/kaggle/working/sigec')
os.chdir('/kaggle/working/sigec')

# 2. Datos: tolerante al nombre
chest = []
for d, ds, fs in os.walk('/kaggle/input'):
    if 'train' in ds and os.path.isdir(os.path.join(d, 'train', 'NORMAL')):
        chest.append(d)
csv = [f for f in glob.glob('/kaggle/input/**/*.csv', recursive=True) if 'show' in os.path.basename(f).lower()]
print('codigo :', SRC); print('rayos X:', chest); print('csv    :', csv)
assert chest, 'Falta el dataset de radiografias: Add Input -> Datasets -> "Chest X-Ray Images (Pneumonia)" (paultimothymooney)'
assert csv, 'Falta el CSV: Add Input -> Datasets -> "Medical Appointment No Shows" (joniarroba)'
CHEST, CSV = chest[0], csv[0]

import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
!pip -q install scikit-learn pandas joblib

## Inasistencia (minutos)

In [ ]:
!python ml/no-show/train.py --csv "$CSV"
!ls -la ml/no-show/models ml/no-show/reports

## Neumonia (~20-40 min con GPU)
Para probar rapido primero: agrega `--subconjunto 200 --epocas 2 --epocas-ajuste 0`.

In [ ]:
!python ml/pneumonia/train.py --datos "$CHEST"
!ls -la ml/pneumonia/models ml/pneumonia/reports

### TFLite para movil / PWA (opcional)

In [ ]:
!python ml/pneumonia/convertir.py --modelo ml/pneumonia/models/pneumonia.keras --tflite --cuantizar
!ls -la ml/pneumonia/models

## Metricas
Neumonia: mira la **sensibilidad**. Inasistencia: recall/F1/AUC.

In [ ]:
import json
for f in sorted(glob.glob('ml/*/reports/**/*.json', recursive=True)):
    print('==', f)
    d = json.load(open(f))
    print(json.dumps(d.get('metricas', d), indent=1, ensure_ascii=False)[:1500])

## Descargar
Empaqueta en `/kaggle/working/resultados.zip`; bajalo desde el panel *Output* (o con *Save Version -> Save & Run All* y luego Output). Descomprimelo en la **raiz** de tu repo local.

In [ ]:
!cd /kaggle/working/sigec && zip -rq /kaggle/working/resultados.zip ml/pneumonia/models ml/pneumonia/reports ml/no-show/models ml/no-show/reports
!ls -la /kaggle/working/resultados.zip